# Brainmaps

Python dependencies: `numpy`, `pandas`, `scipy`, `networkx`, `statsmodels`, `patsy`. Execute the code cells in order. Calculations use only data supplied in memory.

## Inputs and model settings

Provide participant-level regional topology data as in-memory DataFrames. Required biological maps are cell types, gene expression, and receptors; additional normative maps can be supplied. All inputs use 100 Schaefer-Yeo17 parcels. No input files, local paths, or participant data are embedded in this notebook.

In [ ]:
import logging

import numpy as np
import pandas as pd
import networkx as nx
from scipy.stats import pearsonr

LOG = logging.getLogger("petkls.brainmaps")
GROUP_METRICS = {
    "Participation coefficient": "participation_coeff_17Networks_",
    "Betweenness": "betweenness_17Networks_",
    "Clustering coefficient": "clustering_coeff_17Networks_",
    "Strength": "strength_17Networks_",
}
TEMPORAL_METRICS = {**GROUP_METRICS, "Degree": "degree_17Networks_"}
ROI_COLUMNS = ("ROI.Name_17_mod", "ROI.Name_17", "roi", "ROI", "roi_name", "ROI.Name")
SETTINGS = {
    "time_min": -15,
    "time_max": 15,
    "spline_df": 10,
    "spline_degree": 3,
    "centiloid_threshold": 24,
}

In [ ]:
def normalize_roi(name: str) -> str:
    roi = str(name).strip()
    return roi.replace("17Networks_", "").replace("7Networks_", "")


def metric_matrix(data: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = [col for col in data if col.startswith(prefix)]
    matrix = data[cols].apply(pd.to_numeric, errors="coerce")
    matrix.columns = [normalize_roi(col[len(prefix):]) for col in cols]
    return matrix


def pearson_safe(x: np.ndarray, y: np.ndarray) -> tuple[float, int]:
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    keep = np.isfinite(x) & np.isfinite(y)
    n = int(keep.sum())
    if n < 3 or np.ptp(x[keep]) < 1e-12 or np.ptp(y[keep]) < 1e-12:
        return float("nan"), n
    return float(pearsonr(x[keep], y[keep]).statistic), n

In [ ]:
def validate_regional_metrics(data: pd.DataFrame, *, staging: bool) -> pd.DataFrame:
    data = data.copy()
    data.columns = data.columns.str.strip()
    required = {"age", "sex", "estdtt0"} if staging else {
        "cdr_glb_score", "pet_cl_refwc_gca_gaain"
    }
    missing = required.difference(data.columns)
    if missing:
        raise ValueError(f"Missing metadata: {sorted(missing)}")
    metrics = TEMPORAL_METRICS if staging else GROUP_METRICS
    for metric, prefix in metrics.items():
        roi_cols = [col for col in data.columns if col.startswith(prefix)]
        rois = [normalize_roi(col[len(prefix):]) for col in roi_cols]
        if len(roi_cols) != 100 or len(set(rois)) != 100:
            raise ValueError(
                f"{metric}: expected 100 unique Schaefer-100 regional columns; "
                f"found {len(roi_cols)}."
            )
    if "subject_id" in data and data["subject_id"].dropna().duplicated().any():
        raise ValueError("More than one observation per subject.")
    return data


def prepare_biological_maps(maps: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    expected = {"celltype", "gene", "receptor"}
    if not expected.issubset(maps):
        raise ValueError(f"Required biological modalities: {sorted(expected)}")
    prepared = {}
    for modality, supplied in maps.items():
        raw = supplied.copy()
        roi_col = next((name for name in ROI_COLUMNS if name in raw.columns), None)
        if roi_col is not None:
            raw = raw.set_index(roi_col)
        raw.index = raw.index.map(normalize_roi)
        if len(raw.index) != 100 or raw.index.duplicated().any():
            raise ValueError(f"{modality}: expected 100 unique Schaefer-100 parcels.")
        features = {}
        for feature in raw.columns:
            key = feature.lower()
            if any(x in key for x in ("roi.label", "roi_label", "label_17", "label_7", "unnamed")):
                continue
            numeric = pd.to_numeric(raw[feature], errors="coerce")
            if numeric.notna().sum() >= 3:
                features[feature] = numeric.to_numpy(dtype=float)
        if not features:
            raise ValueError(f"{modality}: no numeric biological features.")
        prepared[modality] = pd.DataFrame(features, index=raw.index)
    return prepared

## Regional graph metrics (pre-GAM)

Use `regional_graph_metrics` only when participant-level KLS edges are supplied. If regional metrics already exist, pass them directly to the later functions. The original network weighting convention is retained; no new density threshold is imposed.

In [ ]:
def regional_graph_metrics(
    subject_edges: pd.DataFrame,
    *,
    metadata_columns: tuple[str, ...] = (
        "subject_id", "age", "sex", "estdtt0",
        "cdr_glb_score", "pet_cl_refwc_gca_gaain"
    ),
) -> pd.DataFrame:
    """Compute original weighted regional metrics from participant-wise ROI edges.

    Edge columns use ROI_A-ROI_B names. As in the source notebook,
    the graph weight is the KLS value, including for weighted betweenness.
    This function does not impose a new graph-density threshold.
    """
    edges = {}
    rois = set()
    for name in subject_edges.columns:
        if "-" not in str(name):
            continue
        left, right = str(name).split("-", 1)
        left, right = normalize_roi(left), normalize_roi(right)
        if left == right:
            continue
        pair = tuple(sorted((left, right)))
        if pair in edges:
            raise ValueError(f"Duplicated symmetric ROI pair: {pair}")
        edges[pair] = name
        rois.update(pair)

    rois = sorted(rois)
    if len(rois) != 100 or len(edges) != 4950:
        raise ValueError(f"Expected 100 ROIs and 4,950 undirected edges; got {len(rois)} and {len(edges)}.")
    if subject_edges.empty:
        raise ValueError("No participants supplied.")

    index = {roi: i for i, roi in enumerate(rois)}
    edge_order = list(edges.values())
    edge_coords = [(index[a], index[b]) for a, b in edges]
    weights = subject_edges[edge_order].apply(pd.to_numeric, errors="coerce").to_numpy()
    if not np.isfinite(weights).all():
        raise ValueError("Edge matrix contains missing or nonfinite KLS values.")
    if (weights < 0).any() or (weights > 1).any():
        raise ValueError("KLS edge weights must lie in [0, 1].")
    modules = np.array([roi.split("_")[1] for roi in rois])
    metric_names = ("strength", "degree", "clustering_coeff", "betweenness", "participation_coeff")
    values = {name: np.zeros((len(subject_edges), 100), dtype=float) for name in metric_names}

    for row_no, row in enumerate(weights):
        matrix = np.zeros((100, 100), dtype=float)
        for (i, j), weight in zip(edge_coords, row, strict=True):
            matrix[i, j] = matrix[j, i] = weight
        strength = matrix.sum(axis=1)
        graph = nx.from_numpy_array(matrix)
        clustering = nx.clustering(graph, weight="weight")
        betweenness = nx.betweenness_centrality(graph, weight="weight", normalized=True)
        values["strength"][row_no] = strength
        values["degree"][row_no] = (matrix > 0).sum(axis=1)
        values["clustering_coeff"][row_no] = [clustering[i] for i in range(100)]
        values["betweenness"][row_no] = [betweenness[i] for i in range(100)]
        for i in range(100):
            if strength[i] > 0:
                fractions = [
                    matrix[i, modules == module].sum() / strength[i]
                    for module in np.unique(modules)
                ]
                values["participation_coeff"][row_no, i] = 1 - np.sum(np.square(fractions))
            else:
                values["participation_coeff"][row_no, i] = np.nan

    retained = [col for col in metadata_columns if col in subject_edges]
    result = subject_edges[retained].reset_index(drop=True).copy()
    for metric, matrix in values.items():
        metric_df = pd.DataFrame(
            matrix,
            columns=[f"{metric}_17Networks_{roi}" for roi in rois],
        )
        result = pd.concat([result, metric_df], axis=1)
    return result

# Microstructural gradients

## Age- and sex-adjusted regional GAM trajectories

Each of five regional graph properties (strength, degree, clustering, betweenness, and participation) is modeled against estimated time to amyloid positivity using a Gaussian GAM with cubic B-splines and age/sex covariates. Predictions are evaluated over the configured disease-time window.

In [ ]:
def fit_roi_gams(data: pd.DataFrame, config: dict) -> pd.DataFrame:
    # Keep the GAM dependency isolated from input validation and correlations.
    import statsmodels.api as sm
    from statsmodels.gam.api import BSplines, GLMGam

    base = data.copy()
    base["estdtt0"] = pd.to_numeric(base["estdtt0"], errors="coerce")
    base["age"] = pd.to_numeric(base["age"], errors="coerce")
    base = base.dropna(subset=["estdtt0", "age", "sex"])
    if len(base) < 40:
        raise ValueError("At least 40 participants with time, age and sex are required for GAM fitting.")
    sex = base["sex"].astype(str)
    dummies = pd.get_dummies(sex, prefix="sex", drop_first=True, dtype=float)
    reference_age = float(base["age"].median())
    reference_sex = sex.mode().iat[0]
    grid = np.arange(float(config["time_min"]), float(config["time_max"]) + 1e-9, 1.0)
    results = []

    for metric, prefix in TEMPORAL_METRICS.items():
        roi_matrix = metric_matrix(base, prefix)
        for roi in roi_matrix:
            y = roi_matrix[roi]
            valid = y.notna()
            sub = base.loc[valid]
            if len(sub) < 40 or sub["estdtt0"].nunique() < config["spline_df"] + 1:
                raise ValueError(f"Insufficient usable observations for {metric} / {roi}.")
            time = sub[["estdtt0"]].reset_index(drop=True)
            smooth = BSplines(time, df=[config["spline_df"]], degree=[config["spline_degree"]])
            design = pd.DataFrame({"const": np.ones(len(sub)), "age": sub["age"].to_numpy()})
            sex_part = dummies.loc[valid].reset_index(drop=True)
            design = pd.concat([design, sex_part], axis=1)
            model = GLMGam(
                endog=y.loc[valid].to_numpy(dtype=float),
                exog=design,
                smoother=smooth,
                family=sm.families.Gaussian(),
            )
            fit = model.fit()
            within = grid[(grid >= time["estdtt0"].min()) & (grid <= time["estdtt0"].max())]
            if len(within) < 2:
                raise ValueError(f"No prediction support in requested window for {metric} / {roi}.")
            new_design = pd.DataFrame({"const": np.ones(len(within)), "age": reference_age})
            for dummy in dummies:
                category = dummy.removeprefix("sex_")
                new_design[dummy] = float(category == reference_sex)
            pred = fit.predict(
                exog=new_design,
                exog_smooth=pd.DataFrame({"estdtt0": within}),
            )
            results.extend(
                (metric, roi, float(t), float(value))
                for t, value in zip(within, pred, strict=True)
            )
        LOG.info("Fitted %s: 100 parcels", metric)
    return pd.DataFrame(results, columns=["graph_property", "ROI", "estdtt0", "predicted_value"])

## Spatial correspondence with normative maps

At each estimated disease time, regional predictions are correlated with each biological feature across the 100 parcels. These Pearson correlations are descriptive and are not spatial-null-adjusted significance tests.

In [ ]:
def correlate_trajectories(predictions: pd.DataFrame, brainmaps: dict[str, pd.DataFrame]) -> pd.DataFrame:
    records = []
    for (metric, time), group in predictions.groupby(["graph_property", "estdtt0"], sort=True):
        topo = group.set_index("ROI")["predicted_value"]
        for modality, maps in brainmaps.items():
            common = topo.index.intersection(maps.index)
            if len(common) != 100:
                raise ValueError(f"{metric}/{modality}: {len(common)} parcels matched; expected 100.")
            x = topo.loc[common].to_numpy(dtype=float)
            for feature in maps:
                r, n = pearson_safe(x, maps.loc[common, feature].to_numpy())
                records.append((metric, modality, feature, time, r, n))
    return pd.DataFrame(records, columns=["graph_property", "modality", "feature", "estdtt0", "pearson_r", "n_parcels"])

# Group gradients

## Healthy and early Alzheimer group comparisons

Healthy: CDR 0 and Centiloid below 24. EarlyAD: CDR 0.5 and Centiloid at least 24. Group means are calculated separately for every regional graph property, then correlated with normative maps. Participant counts are taken from the supplied input data.

In [ ]:
def parse_cdr(values: pd.Series) -> pd.Series:
    cleaned = values.astype(str).str.extract(r"^\s*(\d+(?:\.\d+)?)", expand=False)
    return pd.to_numeric(cleaned, errors="coerce")


def group_masks(data: pd.DataFrame, threshold: float) -> dict[str, pd.Series]:
    cdr = parse_cdr(data["cdr_glb_score"])
    centiloid = pd.to_numeric(data["pet_cl_refwc_gca_gaain"], errors="coerce")
    return {
        "Healthy": (cdr == 0) & (centiloid < threshold),
        "EarlyAD": (cdr == 0.5) & (centiloid >= threshold),
    }


def correlate_groups(data: pd.DataFrame, brainmaps: dict[str, pd.DataFrame], cfg: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    masks = group_masks(data, cfg["centiloid_threshold"])
    counts = {
        label: int(data.loc[mask, "subject_id"].nunique() if "subject_id" in data else mask.sum())
        for label, mask in masks.items()
    }
    if any(n < 3 for n in counts.values()):
        raise ValueError(f"Groups are empty or too small: {counts}")
    expected = cfg.get("expected_group_counts")
    if expected and any(counts.get(name) != target for name, target in expected.items()):
        raise ValueError(f"Actual group counts {counts} do not match expected {expected}.")
    records = []
    for metric, prefix in GROUP_METRICS.items():
        matrix = metric_matrix(data, prefix)
        vectors = {name: matrix.loc[mask].mean(axis=0) for name, mask in masks.items()}
        for modality, maps in brainmaps.items():
            common = vectors["Healthy"].index.intersection(maps.index)
            if len(common) != 100:
                raise ValueError(f"{metric}/{modality}: {len(common)} parcels matched; expected 100.")
            for feature in maps:
                r_h, n_h = pearson_safe(vectors["Healthy"].loc[common].to_numpy(), maps.loc[common, feature].to_numpy())
                r_e, n_e = pearson_safe(vectors["EarlyAD"].loc[common].to_numpy(), maps.loc[common, feature].to_numpy())
                records.append((metric, modality, feature, r_h, r_e, r_e - r_h, n_h, n_e))
    result = pd.DataFrame(records, columns=[
        "graph_property", "modality", "feature", "r_Healthy", "r_EarlyAD",
        "delta_r_EarlyAD_minus_Healthy", "n_parcels_Healthy", "n_parcels_EarlyAD",
    ])
    count_df = pd.DataFrame([{"group": name, "n_participants": count} for name, count in counts.items()])
    return result, count_df

## Run and retrieve results

Supply the three in-memory inputs and call the function below. No files are written.

In [ ]:
def run_brainmap_analysis(
    staging_metrics: pd.DataFrame,
    group_metrics: pd.DataFrame,
    biological_maps: dict[str, pd.DataFrame],
    settings: dict | None = None,
) -> dict[str, pd.DataFrame]:
    """Run the computational analysis using in-memory input tables."""
    cfg = SETTINGS | (settings or {})
    staging = validate_regional_metrics(staging_metrics, staging=True)
    groups = validate_regional_metrics(group_metrics, staging=False)
    maps = prepare_biological_maps(biological_maps)
    predictions = fit_roi_gams(staging, cfg)
    temporal = correlate_trajectories(predictions, maps)
    summary = temporal.assign(abs_r=temporal["pearson_r"].abs()).groupby(
        ["graph_property", "modality", "feature"], as_index=False
    ).agg(
        mean_abs_r=("abs_r", "mean"),
        max_abs_r=("abs_r", "max"),
        n_timepoints=("estdtt0", "nunique"),
    )
    group_corr, sample_sizes = correlate_groups(groups, maps, cfg)
    return {
        "regional_gam_predictions": predictions,
        "temporal_spatial_correlations": temporal,
        "temporal_feature_summary": summary,
        "group_spatial_correlations": group_corr,
        "group_sample_sizes": sample_sizes,
    }

In [ ]:
# Required in-memory inputs: staging_metrics, group_metrics, biological_maps
# analysis_results = run_brainmap_analysis(staging_metrics, group_metrics, biological_maps)
# analysis_results.keys()